# 主循环（ver3：相对收益 + MSE 调参 + GMV 对照）

流程：数据 → 单月面板预计算 → 验证期诊断 → 调 alpha（按 MSE）→ 预计算 mu/Σ → 调 gamma（B+LW）→ 测试 E0–E4+GMV → 评估。

**ver3 关键改动**：训练目标为横截面相对收益；alpha 按验证期 MSE 选择（非 IC）；新增 GMV 最小方差对照；E0 与优化策略共用同一候选池口径。

In [1]:
import sys, json, os, pickle
sys.path.insert(0, 'e:/4020prj/code/ver3')
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from src.config import CONFIG
from src import data as data_mod
from src import return_model as rm
from src import risk_model as rk
from src import portfolio as pf
from src import performance as perf
from src.data import build_data, precompute_panels, assemble_train_panel, score_from_panel

OUT = CONFIG['output_dir']
CACHE_DIR = os.path.join(OUT, 'cache')
os.makedirs(OUT, exist_ok=True)


### 工具函数（断点续跑）

In [2]:
def month_periods(start, end):
    return list(pd.period_range(start, end, freq='M'))


def records_to_returns(records):
    return pd.Series([r['nav_mult'] - 1.0 for r in records], index=[r['month'] for r in records])


def _jsonable(obj):
    if isinstance(obj, dict):
        return {k: _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    return obj


def save_json(obj, name):
    with open(os.path.join(OUT, name), 'w', encoding='utf-8') as f:
        json.dump(_jsonable(obj), f, ensure_ascii=False, indent=2)


def load_json(name):
    p = os.path.join(OUT, name)
    return json.load(open(p, encoding='utf-8')) if os.path.exists(p) else None


### `compute_candidates(...)`

每月过滤后的候选（风险窗内历史完整，剔除年内退市/长停牌导致的 NaN）。E0 与所有优化策略共用此口径。

In [3]:
def compute_candidates(R, universe, months, cfg=CONFIG):
    cand_by_month = {}
    for m in months:
        cand = universe.xs(m.year, level='year').index
        risk_months = month_periods(m - cfg['risk_window_months'] - 1, m - 2)
        complete = R.loc[risk_months, cand].notna().all(axis=0)
        cand_by_month[m] = complete[complete].index
    return cand_by_month


### `tune_alpha(...)`（按验证期 MSE 选择，报告 IC 与零基准）

对每个 alpha：逐月拼训练面板 → 拟合 → 预测相对收益 → 计算 `relative_mse`（真实相对收益 = R − 池等权均值）与 IC。选验证期平均 MSE 最小者；同时返回每个 alpha 的 MSE/IC 与「零相对收益」基准 MSE（即预测全 0 的误差）。

In [4]:
def tune_alpha(panels, R, cand_by_month, months, model, fields, cfg=CONFIG):
    results = {}
    best_mse, best_alpha = np.inf, cfg['ridge_grid'][0]
    for a in cfg['ridge_grid']:
        mses, ics = [], []
        for m in months:
            cand = cand_by_month[m]
            if len(cand) == 0:
                continue
            train_months = month_periods(m - cfg['return_window_months'] - 1, m - 2)
            panel = assemble_train_panel(train_months, panels, cfg)
            if panel.empty:
                continue
            score = score_from_panel(panels[m], cand)
            mu, _ = rm.fit_predict_return(panel, score, a, fields)
            r_real = R.loc[m].reindex(cand)
            mses.append(rm.relative_mse(mu, r_real))
            ics.append(rm.cross_sectional_spearman(mu, r_real))
        mean_mse = np.nanmean(mses) if mses else np.inf
        mean_ic = np.nanmean(ics) if ics else np.nan
        results[a] = {'mse': mean_mse, 'ic': mean_ic}
        if mean_mse < best_mse:
            best_mse, best_alpha = mean_mse, a
    zero_mses = []
    for m in months:
        cand = cand_by_month[m]
        if len(cand) == 0:
            continue
        r_rel = R.loc[m].reindex(cand)
        r_rel = r_rel - r_rel.mean()
        zero_mses.append(float((r_rel ** 2).mean()))
    zero_mse = np.nanmean(zero_mses) if zero_mses else np.nan
    return best_alpha, results, zero_mse


### `compute_mu` / `compute_sigma`（预计算预测与风险，供 gamma 调参和测试复用）

B/BQ 用冻结 alpha 预测相对收益（回退股票填 0）；H 为历史均值（绝对收益基准）。Sigma 用截至 m-2 的 60 个月收益。

In [5]:
def compute_mu(panels, R, cand_by_month, months, model, alpha, cfg=CONFIG):
    fields = cfg['BASE_FIELDS'] if model == 'B' else cfg['BASE_FIELDS'] + cfg['ENHANCED_FIELDS']
    out = {}
    for m in months:
        cand = cand_by_month[m]
        train_months = month_periods(m - cfg['return_window_months'] - 1, m - 2)
        if model == 'H':
            mu = rm.historical_mean_prediction(R.loc[train_months, cand]).reindex(cand)
        else:
            panel = assemble_train_panel(train_months, panels, cfg)
            score = score_from_panel(panels[m], cand)
            mu, _ = rm.fit_predict_return(panel, score, alpha, fields)
        out[m] = mu
    return out


def compute_sigma(R, cand_by_month, months, risk_method, cfg=CONFIG):
    out = {}
    for m in months:
        cand = cand_by_month[m]
        risk_months = month_periods(m - cfg['risk_window_months'] - 1, m - 2)
        Sigma, _ = rk.estimate_covariance(R.loc[risk_months, cand].values, risk_method)
        out[m] = Sigma
    return out


### `equal_weight_benchmark` / `backtest` / `tune_gamma`

E0 与优化策略共用 `cand_by_month`（60 个月风险历史完整）；`backtest` 支持 `gmv=True` 走 `solve_gmv`、`keep_weights=True` 保存每月权重供权重距离对比；`tune_gamma` 用**基础模型 B+LW**在验证期选择 gamma，所有优化策略共用。

In [6]:
def equal_weight_benchmark(R, cand_by_month, months, cfg=CONFIG):
    nav, w_prev = 1.0, None
    records = []
    for m in months:
        cand = cand_by_month[m]
        if len(cand) == 0:
            continue
        w_target = pd.Series(1.0 / len(cand), index=cand)
        w_pre = pd.Series(0.0, index=cand) if w_prev is None else pf.drift_weights(w_prev, R.loc[m-1].reindex(w_prev.index))
        all_a = w_target.index.union(w_pre.index)
        acct = pf.execute_and_account(w_target.reindex(all_a, fill_value=0.0), w_pre.reindex(all_a, fill_value=0.0), R.loc[m].reindex(all_a, fill_value=0.0))
        nav *= acct['nav_mult']
        records.append({'month': m, 'nav': nav, **acct})
        w_prev = w_target
    return records


def backtest(mu_by_month, sigma_by_month, R, industry, cand_by_month, months, gamma, *, gmv=False, keep_weights=False, cfg=CONFIG):
    nav, w_prev = 1.0, None
    records = []
    for m in months:
        cand = cand_by_month[m]
        if len(cand) == 0:
            continue
        if gmv:
            w_target, status, _ = pf.solve_gmv(sigma_by_month[m], industry.reindex(cand), cfg)
        else:
            w_target, status, _ = pf.solve_portfolio(mu_by_month[m], sigma_by_month[m], industry.reindex(cand), gamma, None, None, cfg)
        w_pre = pd.Series(0.0, index=cand) if w_prev is None else pf.drift_weights(w_prev, R.loc[m-1].reindex(w_prev.index))
        all_a = w_target.index.union(w_pre.index)
        acct = pf.execute_and_account(w_target.reindex(all_a, fill_value=0.0), w_pre.reindex(all_a, fill_value=0.0), R.loc[m].reindex(all_a, fill_value=0.0))
        nav *= acct['nav_mult']
        rec = {'month': m, 'nav': nav, 'status': status, **acct}
        if keep_weights:
            rec['w_target'] = w_target
        records.append(rec)
        w_prev = w_target
    return records


def tune_gamma(mu_b, sigma_lw, R, industry, cand_by_month, months, rf, cfg=CONFIG):
    best_sharpe, best_gamma = -np.inf, cfg['gamma_grid'][0]
    for g in cfg['gamma_grid']:
        records = backtest(mu_b, sigma_lw, R, industry, cand_by_month, months, gamma=g, gmv=False, cfg=cfg)
        r_net = records_to_returns(records)
        sharpe = perf.performance_metrics(r_net, rf.reindex(r_net.index))['sharpe']
        if sharpe > best_sharpe:
            best_sharpe, best_gamma = sharpe, g
    return best_gamma, best_sharpe


### `run_validation_diagnostics(...)`（验证期诊断）

逐字段单变量 IC、B/BQ 逐月 IC 及差值、预测横截面标准差、回归系数均值、B/BQ 预测相关性、回退比例、B/BQ 的 MSE。用于回答「增强是否改善排序/预测、预测是否被压缩、多少预测来自岭回归」。

In [7]:
def run_validation_diagnostics(panels, R, cand_by_month, val_months, alpha_by_model, cfg=CONFIG):
    all_fields = cfg['BASE_FIELDS'] + cfg['ENHANCED_FIELDS']
    diag = {'field_ic': {f: [] for f in all_fields}, 'b_ic': [], 'bq_ic': [],
            'pred_std_B': [], 'pred_std_BQ': [], 'fallback_ratio': [],
            'bq_b_corr': [], 'b_mse': [], 'bq_mse': []}
    coef_B, coef_BQ = [], []
    for m in val_months:
        cand = cand_by_month[m]
        if len(cand) == 0:
            continue
        train_months = month_periods(m - cfg['return_window_months'] - 1, m - 2)
        score = score_from_panel(panels[m], cand)
        r_real = R.loc[m].reindex(cand)
        r_rel = r_real - r_real.mean()
        for f in all_fields:
            both = pd.concat([score[f].rename('x'), r_rel.rename('r')], axis=1).dropna()
            if len(both) >= 3:
                diag['field_ic'][f].append(spearmanr(both['x'], both['r']).correlation)
        panel = assemble_train_panel(train_months, panels, cfg)
        if panel.empty:
            continue
        mu_b, model_b = rm.fit_predict_return(panel, score, alpha_by_model['B'], cfg['BASE_FIELDS'])
        mu_bq, model_bq = rm.fit_predict_return(panel, score, alpha_by_model['BQ'], cfg['BASE_FIELDS'] + cfg['ENHANCED_FIELDS'])
        diag['b_ic'].append(rm.cross_sectional_spearman(mu_b, r_real))
        diag['bq_ic'].append(rm.cross_sectional_spearman(mu_bq, r_real))
        diag['pred_std_B'].append(float(mu_b.std()))
        diag['pred_std_BQ'].append(float(mu_bq.std()))
        diag['fallback_ratio'].append(float(score['fallback'].mean()))
        diag['bq_b_corr'].append(float(np.corrcoef(mu_b, mu_bq)[0, 1]))
        diag['b_mse'].append(rm.relative_mse(mu_b, r_real))
        diag['bq_mse'].append(rm.relative_mse(mu_bq, r_real))
        coef_B.append(model_b.coef_)
        coef_BQ.append(model_bq.coef_)
    summary = {
        'field_ic_mean': {f: float(np.nanmean(v)) for f, v in diag['field_ic'].items()},
        'b_ic_mean': float(np.nanmean(diag['b_ic'])),
        'bq_ic_mean': float(np.nanmean(diag['bq_ic'])),
        'ic_diff_mean': float(np.nanmean(np.array(diag['bq_ic']) - np.array(diag['b_ic']))),
        'pred_std_B_mean': float(np.nanmean(diag['pred_std_B'])),
        'pred_std_BQ_mean': float(np.nanmean(diag['pred_std_BQ'])),
        'fallback_ratio_mean': float(np.nanmean(diag['fallback_ratio'])),
        'bq_b_corr_mean': float(np.nanmean(diag['bq_b_corr'])),
        'b_mse_mean': float(np.nanmean(diag['b_mse'])),
        'bq_mse_mean': float(np.nanmean(diag['bq_mse'])),
        'coef_B_mean': np.nanmean(coef_B, axis=0).tolist() if coef_B else None,
        'coef_BQ_mean': np.nanmean(coef_BQ, axis=0).tolist() if coef_BQ else None,
    }
    return summary, diag


### `freeze_config` / `evaluate` / `weight_distance`

In [8]:
def freeze_config(alpha_by_model, gamma, cfg=CONFIG):
    frozen = {'alpha_by_model': alpha_by_model, 'gamma': gamma,
              'fields': cfg['BASE_FIELDS'] + cfg['ENHANCED_FIELDS'],
              'time_split': {'validation': [cfg['validation_start'], cfg['validation_end']],
                             'freeze_month': cfg['freeze_month'],
                             'test': [cfg['test_start'], cfg['test_end']]}}
    with open(os.path.join(OUT, 'frozen_config.json'), 'w', encoding='utf-8') as f:
        json.dump(frozen, f, ensure_ascii=False, indent=2)
    return frozen


def evaluate(results, rf, cfg=CONFIG):
    rets = {k: records_to_returns(v) for k, v in results.items()}
    table = pd.DataFrame({k: perf.performance_metrics(rets[k], rf.reindex(rets[k].index)) for k in rets}).T
    boot = {}
    for a, b in [('E4', 'E0'), ('E4', 'E3'), ('E4', 'GMV'), ('E3', 'GMV')]:
        common = rets[a].index.intersection(rets[b].index)
        ra = rets[a].reindex(common)
        rb = rets[b].reindex(common)
        rf_c = rf.reindex(common)
        # rf 已 ffill 到 test_end，此处再剔除缺失月份作防御，避免 numpy 遇 NaN
        valid = ra.notna() & rb.notna() & rf_c.notna()
        boot[f'{a}-{b}'] = perf.circular_block_bootstrap(ra[valid], rb[valid], rf=rf_c[valid])
    ff5 = pd.read_csv(cfg['factor_benchmarks'][0], parse_dates=['date']).set_index('date'); ff5.index = ff5.index.to_period('M')
    hxz = pd.read_csv(cfg['factor_benchmarks'][1], parse_dates=['date']).set_index('date'); hxz.index = hxz.index.to_period('M')
    r_excess = rets['E4'] - rf.reindex(rets['E4'].index)
    reg = {}
    for name, F in [('FF5', ff5), ('HXZ', hxz)]:
        common = r_excess.index.intersection(F.index)
        r_e, F_c = r_excess.reindex(common), F.reindex(common)
        mask = r_e.notna() & F_c.notna().all(axis=1)
        reg[name] = perf.factor_regression(r_e[mask], F_c[mask])
    return {'returns': rets, 'table': table, 'bootstrap': boot, 'factor_reg': reg}


def weight_distance(records_a, records_b):
    dists = []
    for ra, rb in zip(records_a, records_b):
        if ra['month'] != rb['month']:
            continue
        union = ra['w_target'].index.union(rb['w_target'].index)
        dists.append(float((ra['w_target'].reindex(union, fill_value=0.0) - rb['w_target'].reindex(union, fill_value=0.0)).abs().sum()))
    return float(np.nanmean(dists)) if dists else np.nan


### `main()` —— 端到端（ver3）

In [9]:
def main():
    data = build_data()
    R, D, universe, industry, rf = data['R'], data['D'], data['universe'], data['industry'], data['rf']
    val_months = month_periods(CONFIG['validation_start'], CONFIG['validation_end'])
    test_months = month_periods(CONFIG['test_start'], CONFIG['test_end'])
    target_months = val_months + test_months

    cand_by_month = compute_candidates(R, universe, target_months)

    first_label = val_months[0] - (CONFIG['return_window_months'] + 1)
    label_months = month_periods(first_label, test_months[-1])
    panels = precompute_panels(D, R, universe, label_months, CACHE_DIR)
    print('panels:', len(panels), '个月 | candidates:', {k: len(v) for k, v in list(cand_by_month.items())[:1]})

    # 1) 调 alpha（MSE）
    alpha_by_model, alpha_results = {}, {}
    for model, fields in [('B', CONFIG['BASE_FIELDS']), ('BQ', CONFIG['BASE_FIELDS'] + CONFIG['ENHANCED_FIELDS'])]:
        a, res, zero_mse = tune_alpha(panels, R, cand_by_month, val_months, model, fields)
        alpha_by_model[model] = a
        alpha_results[model] = {'grid': res, 'zero_mse': zero_mse}
        print(f'alpha[{model}] = {a}  (MSE 选) | 零基准 MSE {zero_mse:.6f}')
        for av, r in res.items():
            mse = r['mse']
            ic = r['ic']
            print(f'   alpha={av}: MSE={mse:.6f}  IC={ic:.4f}')
    save_json({'alpha_by_model': alpha_by_model, 'alpha_results': alpha_results}, 'alpha_by_model.json')

    # 2) 预计算 mu / Sigma
    mu_cache = {'H': compute_mu(panels, R, cand_by_month, target_months, 'H', None),
                'B': compute_mu(panels, R, cand_by_month, target_months, 'B', alpha_by_model['B']),
                'BQ': compute_mu(panels, R, cand_by_month, target_months, 'BQ', alpha_by_model['BQ'])}
    sigma_cache = {'S': compute_sigma(R, cand_by_month, target_months, 'S'),
                   'LW': compute_sigma(R, cand_by_month, target_months, 'LW')}

    # 3) 验证期诊断
    diag_summary, _ = run_validation_diagnostics(panels, R, cand_by_month, val_months, alpha_by_model)
    save_json(diag_summary, 'validation_diagnostics.json')
    print('--- 验证期诊断 ---')
    print('逐字段 IC:', {k: round(v, 4) for k, v in diag_summary['field_ic_mean'].items()})
    print('B IC %.4f | BQ IC %.4f | IC差 %.4f' % (diag_summary['b_ic_mean'], diag_summary['bq_ic_mean'], diag_summary['ic_diff_mean']))
    print('预测 std: B %.6f | BQ %.6f' % (diag_summary['pred_std_B_mean'], diag_summary['pred_std_BQ_mean']))
    print('回退比例 %.4f | B/BQ 预测相关 %.4f' % (diag_summary['fallback_ratio_mean'], diag_summary['bq_b_corr_mean']))
    print('MSE: B %.6f | BQ %.6f | 零基准 %.6f' % (diag_summary['b_mse_mean'], diag_summary['bq_mse_mean'], alpha_results['B']['zero_mse']))

    # 4) 调 gamma（B+LW）
    gamma, gs = tune_gamma(mu_cache['B'], sigma_cache['LW'], R, industry, cand_by_month, val_months, rf)
    save_json({'gamma': gamma, 'val_sharpe': gs}, 'gamma.json')
    print(f'gamma = {gamma}  (B+LW 验证 Sharpe {gs:.4f})')

    # 5) 冻结
    frozen = freeze_config(alpha_by_model, gamma)

    # 6) 测试 E0-E4 + GMV
    strategies = [
        ('E0', lambda: equal_weight_benchmark(R, cand_by_month, test_months)),
        ('E1', lambda: backtest(mu_cache['H'], sigma_cache['S'], R, industry, cand_by_month, test_months, gamma)),
        ('E2', lambda: backtest(mu_cache['B'], sigma_cache['S'], R, industry, cand_by_month, test_months, gamma)),
        ('E3', lambda: backtest(mu_cache['B'], sigma_cache['LW'], R, industry, cand_by_month, test_months, gamma, keep_weights=True)),
        ('E4', lambda: backtest(mu_cache['BQ'], sigma_cache['LW'], R, industry, cand_by_month, test_months, gamma, keep_weights=True)),
        ('GMV', lambda: backtest(mu_cache['B'], sigma_cache['LW'], R, industry, cand_by_month, test_months, gamma, gmv=True, keep_weights=True)),
    ]
    results = {}
    for name, fn in strategies:
        rec = os.path.join(OUT, f'test_records_{name}.pkl')
        if os.path.exists(rec):
            with open(rec, 'rb') as f:
                results[name] = pickle.load(f)
        else:
            results[name] = fn()
            with open(rec, 'wb') as f:
                pickle.dump(results[name], f)

    # 7) 评估
    ev = evaluate(results, rf)
    ev['table'].to_csv(os.path.join(OUT, 'performance_table.csv'))
    print(ev['table'].round(4))
    print('bootstrap:', {k: tuple(tuple(round(x, 4) for x in ci) for ci in v) for k, v in ev['bootstrap'].items()})
    print('factor alpha/R2:', {k: (round(v['alpha_annual'], 4), round(v['r2'], 4)) for k, v in ev['factor_reg'].items()})

    # 8) 测试期 IC（B/BQ）
    test_ic = {}
    for model in ['B', 'BQ']:
        ics = [rm.cross_sectional_spearman(mu_cache[model][m], R.loc[m].reindex(cand_by_month[m])) for m in test_months]
        test_ic[model] = float(np.nanmean(ics))
    print('测试期 IC: B %.4f | BQ %.4f' % (test_ic['B'], test_ic['BQ']))
    save_json({'test_ic': test_ic}, 'test_ic.json')

    # 9) GMV vs E3/E4 对比（换手、权重距离）
    for name in ['E3', 'E4', 'GMV']:
        to = np.nanmean([r['turnover_one_way'] for r in results[name]])
        print(f'{name} 平均单边换手: {to:.4f}')
    print('E3-GMV 权重距离: %.4f' % weight_distance(results['E3'], results['GMV']))
    print('E4-GMV 权重距离: %.4f' % weight_distance(results['E4'], results['GMV']))

    return {'bundle': data, 'frozen': frozen, 'results': results, 'eval': ev}


运行主循环：

In [10]:
if __name__ == '__main__':
    main()


panels: 253 个月 | candidates: {Period('2015-01', 'M'): 100}
alpha[B] = 100  (MSE 选) | 零基准 MSE 0.008718
   alpha=0.1: MSE=0.008709  IC=0.0507
   alpha=1: MSE=0.008709  IC=0.0511
   alpha=10: MSE=0.008708  IC=0.0519
   alpha=100: MSE=0.008708  IC=0.0531
   alpha=1000: MSE=0.008715  IC=0.0523
alpha[BQ] = 100  (MSE 选) | 零基准 MSE 0.008718
   alpha=0.1: MSE=0.008705  IC=0.0445
   alpha=1: MSE=0.008705  IC=0.0447
   alpha=10: MSE=0.008704  IC=0.0451
   alpha=100: MSE=0.008704  IC=0.0466
   alpha=1000: MSE=0.008713  IC=0.0497
--- 验证期诊断 ---
逐字段 IC: {'total_mv': 0.0142, 'book_value_to_total_mktcap_mrq': -0.0347, 'gross_profit_to_asset_ttm': 0.0823, 'asset_growth_mrq': 0.0418, 'momentum_academic_252_21': 0.0404, 'illiq_252': -0.0263, 'total_vol_21': -0.0249, 'accrual_to_asset_ttm': 0.0186, 'net_profit_to_asset_q_var': 0.0219, 'gross_profit_to_asset_q_trend': 0.0035}
B IC 0.0531 | BQ IC 0.0466 | IC差 -0.0065
预测 std: B 0.001952 | BQ 0.002556
回退比例 0.0265 | B/BQ 预测相关 0.7352
MSE: B 0.008708 | BQ 0.008704